# TSA Chapter 12: Smoothed Periodogram: Daniell Kernel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/TSA/blob/main/TSA_ch12/TSA_ch12_smoothing/TSA_ch12_smoothing.ipynb)

In [ ]:
!pip install matplotlib numpy scipy statsmodels pywt -q

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import periodogram

In [ ]:
# Style configuration
COLORS = {
    'blue':   '#1A3A6E',
    'red':    '#DC3545',
    'green':  '#2E7D32',
    'orange': '#E67E22',
    'gray':   '#666666',
    'purple': '#8E44AD',
}

plt.rcParams.update({
    'axes.facecolor':     'none',
    'figure.facecolor':   'none',
    'savefig.transparent': True,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.grid':          False,
    'font.size':          9,
    'axes.labelsize':     9,
    'axes.titlesize':     10,
    'legend.fontsize':    8,
    'xtick.labelsize':    8,
    'ytick.labelsize':    8,
})

In [ ]:
# Chart: ch12_smoothing_comparison
# Daniell kernel smoothing with increasing bandwidth M
def daniell_smooth(psd, M):
    kernel = np.ones(2*M+1) / (2*M+1)
    return np.convolve(psd, kernel, mode='same')

rng = np.random.default_rng(1)
phi = 0.6
N = 1024
eps = rng.normal(0, 1, N+100)
x = np.zeros(N+100)
for t in range(1, N+100):
    x[t] = phi*x[t-1] + eps[t]
x = x[100:]
true_psd = lambda f: 1.0 / (1 - 2*phi*np.cos(2*np.pi*f) + phi**2)

freqs, raw = periodogram(x)
f_true = np.linspace(0.01, 0.49, 400)

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(freqs[1:], raw[1:], color=COLORS['gray'], lw=0.7, alpha=0.5, label='Raw periodogram')
for M, col in [(5, COLORS['blue']), (15, COLORS['orange']), (30, COLORS['red'])]:
    smoothed = daniell_smooth(raw, M)
    ax.semilogy(freqs[1:], smoothed[1:], color=col, lw=1.8, label=f'Daniell M={M}')
ax.semilogy(f_true, true_psd(f_true), 'k--', lw=1.5, label='True PSD')
ax.set_xlabel('Frequency'); ax.set_ylabel('PSD (log scale)')
ax.set_title('Daniell Kernel Smoothing — Bias-Variance Tradeoff')
fig.patch.set_alpha(0); ax.set_facecolor('none')
ax.spines[['top','right']].set_visible(False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=5, frameon=False)
plt.tight_layout()
plt.savefig('ch12_smoothing_comparison.pdf', bbox_inches='tight', dpi=150)
plt.savefig('ch12_smoothing_comparison.png', bbox_inches='tight', dpi=150)
plt.show()